# 07 — FastAPI Serving Layer

Goal: validate production inference contract and endpoint behavior.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pipeline').exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / 'pipeline').exists():
            PROJECT_ROOT = parent
            break
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)

Project root: /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/MLOps, UI, and Deployment/Deploy a Machine Learning Model with Docker


## 7.1 API Design for Inference
**Definition:** Inference API must be explicit, validated, observable, and backward compatible.

**Theory:** Explain mechanism, assumptions, and where this fits in deployment lifecycle.

**Motivation:** Why this matters for reliability, reproducibility, and operations.

**Real-World Example:** Batch endpoint naming migration keeps old clients working while new contract is introduced.

**Visual Explanation:** See figure/code cell below.

**Code Explanation:** Code cell demonstrates concrete implementation details.

**Best Practices:** Version payload contracts and enforce strict validation.

**Common Mistakes:** Allowing loosely typed payloads that silently coerce bad values.

In [2]:
import asyncio
import httpx
from app.main import app

sample = {
    'MedInc': 8.3, 'HouseAge': 42.0, 'AveRooms': 6.9, 'AveBedrms': 1.0,
    'Population': 230.0, 'AveOccup': 3.2, 'Latitude': 37.9, 'Longitude': -122.2,
}

async def smoke():
    transport = httpx.ASGITransport(app=app)
    async with httpx.AsyncClient(transport=transport, base_url='http://test') as client:
        print('health:', (await client.get('/health')).json())
        print('model-info keys:', list((await client.get('/model-info')).json().keys()))
        print('predict:', (await client.post('/predict', json=sample)).json())
        print('predict-batch:', (await client.post('/predict-batch', json={'instances': [sample]})).json())
        print('batch-predict alias:', (await client.post('/batch-predict', json={'instances': [sample]})).json())
        explain = await client.post('/explain', json={'input': sample})
        print('explain keys:', list(explain.json().keys()))

await smoke()

{"timestamp": "2026-06-24T22:04:55.323215+00:00", "level": "INFO", "name": "app.utils", "message": "Health check"}


health: {'status': 'healthy'}
model-info keys: ['model_name', 'best_model_source', 'features', 'metrics', 'profile', 'training_timestamp', 'version']


predict: {'predicted_value': 3.9869}
predict-batch: {'predictions': [3.9869]}
batch-predict alias: {'predictions': [3.9869]}


explain keys: ['shap_values', 'base_value', 'prediction']


## 7.2 Validation and Error Handling
**Definition:** Pydantic constraints reject malformed data before model invocation.

**Theory:** Explain mechanism, assumptions, and where this fits in deployment lifecycle.

**Motivation:** Why this matters for reliability, reproducibility, and operations.

**Real-World Example:** Latitude outside California bounds is rejected at schema layer.

**Visual Explanation:** See figure/code cell below.

**Code Explanation:** Code cell demonstrates concrete implementation details.

**Best Practices:** Validate early and return clear HTTP errors.

**Common Mistakes:** Letting model crash with stack trace due to weak input checks.

In [3]:
import asyncio
import httpx
from app.main import app

async def invalid_payload_check():
    transport = httpx.ASGITransport(app=app)
    async with httpx.AsyncClient(transport=transport, base_url='http://test') as client:
        bad = await client.post('/predict', json={'MedInc': 'bad'})
        print('Status for invalid payload:', bad.status_code)
        print('Error keys:', bad.json().keys())

await invalid_payload_check()

Status for invalid payload: 422
Error keys: dict_keys(['detail'])
